# Context Length Evaluator

Measures how **response latency and token counts scale with increasing context length**.

For each (context, prompt) pair the evaluator:
- Sends a chat request and records wall-clock latency
- Captures prompt and completion tokens from the usage field

The evaluator is **data-agnostic**: it accepts any `Dict[str, str]` mapping a label to a
context string. The `context_utils` module provides helpers to build that dict from any source.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelContextLengthEvaluator
from bellmira.utils.context_utils import (
    contexts_from_word_counts,  # generic — any text string
    contexts_from_files,        # one file per context
    contexts_from_bible,        # convenience wrapper for the bundled Bible corpus
)

## 2. Build contexts

Pick **one** of the three approaches below depending on your data.

### Option A — word-count truncation (any text)

Load any plain text file and slice it at word-count boundaries to produce
contexts of ~1 k, 2 k, 4 k, and 8 k words.

In [ ]:
from bellmira.utils.context_utils import read_text_file

MY_TEXT_PATH = "/workspaces/BeLLMira/resources/data/text/bible.txt"  # replace with any .txt
text = read_text_file(MY_TEXT_PATH)

contexts = contexts_from_word_counts(
    text,
    word_counts=[500, 1000, 2000, 4000],
    # labels=["500w", "1k", "2k", "4k"],  # optional — defaults to "500w", "1000w", ...
)
print({k: f"{len(v.split())} words" for k, v in contexts.items()})

### Option B — one file per context

Each file becomes one context.  The label defaults to the file stem.

In [ ]:
# contexts = contexts_from_files([
#     "/path/to/short_doc.txt",
#     "/path/to/medium_doc.txt",
#     "/path/to/long_doc.txt",
# ])

### Option C — Bible corpus (bundled)

Preserves the original chapter-based slicing from earlier versions.
Each chapter number marks the *start* of the next chapter;
the context is everything **before** it.

In [ ]:
# contexts = contexts_from_bible(
#     bible_path="/workspaces/BeLLMira/resources/data/text/bible.txt",
#     chapter_numbers=[2, 5, 9, 14, 22, 35],
# )
# print(list(contexts.keys()))

## 3. Configuration

In [ ]:
MODEL_URL = "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/"

PROMPTS = [
    "What is the main theme of the text above?",
    "Summarise the key events described in this passage.",
]

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelContextLengthEvaluator(
    url=MODEL_URL,
    prompts=PROMPTS,
    contexts=contexts,
    temperature=0.0,
)

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate(max_prompts=2)
results

## 6. Compute per-context averages

In [ ]:
averages = evaluator.compute_averages(results)
for ctx_key, avg in averages.items():
    print(f"{ctx_key}: exec={avg.get('Avg_execution_time')}s  "
          f"prompt_tok={avg.get('Avg_prompt_tokens')}  "
          f"compl_tok={avg.get('Avg_completion_tokens')}")

## 7. Extract flat metrics for logging

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)
for key, val in metrics.items():
    print(f"{key}: {val}")

## 8. Run via Experiments (multi-model)

Pass the same `contexts` dict to multiple model URLs via `Experiments`.

In [ ]:
from bellmira.llm_experiments.experiments import Experiments

evaluator_args = {
    "prompts": PROMPTS,
    "contexts": contexts,
}

experiments_list = [
    {
        "model": "Qwen2.5 QA",
        "url": "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/",
        "env": "QA",
        "vllm_version": "0.9.2",
        "evaluator": ModelContextLengthEvaluator,
        "evaluator_args": evaluator_args,
        "output_filename": "context_length_results.csv",
    },
    {
        "model": "Qwen2.5 IN",
        "url": "https://millmchatqwen25-backend.dat-aip-millm.in.mbcp.cloud/",
        "env": "IN",
        "vllm_version": "0.9.2",
        "evaluator": ModelContextLengthEvaluator,
        "evaluator_args": evaluator_args,
        "output_filename": "context_length_results.csv",
    },
]

experiments = Experiments(experiments_list, vllm_version="0.9.2")
experiments.create_experiments()
experiments.launch_experiments()